In [1]:
import math
import os
import random
import sys
from typing import List

from dotenv import load_dotenv
from pymilvus import DataType, MilvusClient


In [2]:
OLD_COLLECTION = "BoldSearcher_v2"
NEW_COLLECTION = "BoldSearcher_v3"


In [3]:
def connect_to_zilliz() -> MilvusClient:
    """
    Đọc URI và token từ file .env rồi kết nối Zilliz Cloud.
    """
    load_dotenv()

    uri = os.getenv("ZILLIZ_URI")
    token = os.getenv("ZILLIZ_TOKEN")

    if not uri:
        raise ValueError(
            "Thiếu ZILLIZ_URI. Hãy thêm ZILLIZ_URI vào file .env."
        )

    if not token:
        raise ValueError(
            "Thiếu ZILLIZ_TOKEN. Hãy thêm ZILLIZ_TOKEN vào file .env."
        )

    # MilvusClient là lớp kết nối với Zilliz Cloud. Bạn có thể sử dụng client này để tạo collection, chèn dữ liệu, tìm kiếm vector, v.v.
    client = MilvusClient(
        uri=uri,
        token=token,
    )

    return client


In [4]:
client = connect_to_zilliz()

print("Kết nối Zilliz Cloud thành công.")



Kết nối Zilliz Cloud thành công.


In [5]:
old_schema = client.describe_collection(
    collection_name=OLD_COLLECTION
)

for field in old_schema["fields"]:
    print(field["name"], field["type"])


id 5
video_id 21
frame_id 5
shot_id 5
embedding 101
ocr_text 21
ocr_sparse 104
asr_text 21
asr_sparse 104


# Define Schema 

In [6]:
from pymilvus import (
    MilvusClient,
    DataType,
    Function,
    FunctionType,
)

VISUAL_DIM = 1024

TEXT_ANALYZER_PARAMS = {
    "tokenizer": {
        "type": "language_identifier",
        "identifier": "whatlang",
        "analyzers": {
            "default": {
                "tokenizer": "icu",
                "filter": ["lowercase", "removepunct"],
            },
            "English": {"type": "english"},
            "Vietnamese": {
                "tokenizer": "icu",
                "filter": ["lowercase", "removepunct"],
            },
        },
    }
}

schema = client.create_schema(
    auto_id=True,
    enable_dynamic_field=False,
)

schema.add_field(
    field_name="id",
    datatype=DataType.INT64,
    is_primary=True,
    auto_id=True,
)

schema.add_field(
    field_name="video_id",
    datatype=DataType.VARCHAR,
    max_length=255,
)

schema.add_field(
    field_name="frame_id",
    datatype=DataType.INT64,
)

schema.add_field(
    field_name="shot_id",
    datatype=DataType.INT64,
)

schema.add_field(
    field_name="embedding",
    datatype=DataType.FLOAT_VECTOR,
    dim=VISUAL_DIM,
)

schema.add_field(
    field_name="ocr_text",
    datatype=DataType.VARCHAR,
    max_length=65535,
    enable_analyzer=True,
    analyzer_params=TEXT_ANALYZER_PARAMS,
)

schema.add_field(
    field_name="ocr_sparse",
    datatype=DataType.SPARSE_FLOAT_VECTOR,
)

schema.add_field(
    field_name="asr_text",
    datatype=DataType.VARCHAR,
    max_length=65535,
    enable_analyzer=True,
    analyzer_params=TEXT_ANALYZER_PARAMS,
)

schema.add_field(
    field_name="asr_sparse",
    datatype=DataType.SPARSE_FLOAT_VECTOR,
)


{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'video_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 255}}, {'name': 'frame_id', 'description': '', 'type': <DataType.INT64: 5>}, {'name': 'shot_id', 'description': '', 'type': <DataType.INT64: 5>}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}}, {'name': 'ocr_text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535, 'enable_analyzer': True, 'analyzer_params': '{"tokenizer":{"type":"language_identifier","identifier":"whatlang","analyzers":{"default":{"tokenizer":"icu","filter":["lowercase","removepunct"]},"English":{"type":"english"},"Vietnamese":{"tokenizer":"icu","filter":["lowercase","removepunct"]}}}}'}}, {'name': 'ocr_sparse', 'description': '', 'type': <DataType.SPARSE_FLOAT_VECTOR: 104>}, {'name': 'asr

# Define BM25 Function for OCR, ASR

In [7]:
ocr_bm25 = Function(
    name="ocr_bm25_function",
    function_type=FunctionType.BM25,
    input_field_names=["ocr_text"],
    output_field_names=["ocr_sparse"],
)

asr_bm25 = Function(
    name="asr_bm25_function",
    function_type=FunctionType.BM25,
    input_field_names=["asr_text"],
    output_field_names=["asr_sparse"],
)

schema.add_function(ocr_bm25)
schema.add_function(asr_bm25)


{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'video_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 255}}, {'name': 'frame_id', 'description': '', 'type': <DataType.INT64: 5>}, {'name': 'shot_id', 'description': '', 'type': <DataType.INT64: 5>}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}}, {'name': 'ocr_text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535, 'enable_analyzer': True, 'analyzer_params': '{"tokenizer":{"type":"language_identifier","identifier":"whatlang","analyzers":{"default":{"tokenizer":"icu","filter":["lowercase","removepunct"]},"English":{"type":"english"},"Vietnamese":{"tokenizer":"icu","filter":["lowercase","removepunct"]}}}}'}}, {'name': 'ocr_sparse', 'description': '', 'type': <DataType.SPARSE_FLOAT_VECTOR: 104>, 'is_function_o

# Create Index


In [8]:
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_name="visual_index",
    index_type="AUTOINDEX",
    metric_type="COSINE",
)

index_params.add_index(
    field_name="ocr_sparse",
    index_name="ocr_bm25_index",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={
        "inverted_index_algo": "DAAT_MAXSCORE",
        "bm25_k1": 1.8, # Tham số k1 trong công thức BM25, thường được đặt trong khoảng từ 1.2 đến 2.0. Giá trị này điều chỉnh mức độ bão hòa của điểm số khi tần suất từ khóa tăng lên.
        "bm25_b": 0.75 # Tham số b thuộc [0,1] trong công thức BM25, thường được đặt trong khoảng từ 0.5 đến 0.8. Giá trị này điều chỉnh mức độ ảnh hưởng của độ dài tài liệu đến điểm số.
    }
)

index_params.add_index(
    field_name="asr_sparse",
    index_name="asr_bm25_index",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={
        "inverted_index_algo": "DAAT_MAXSCORE",
        "bm25_k1": 1.8,
        "bm25_b": 0.75
    }
)


# Initialize Collection with new Schema

In [9]:
if client.has_collection(NEW_COLLECTION):
    print(f"{NEW_COLLECTION} đã tồn tại")
else:
    client.create_collection(
        collection_name=NEW_COLLECTION,
        schema=schema,
        index_params=index_params,
    )

    print(f"Đã tạo {NEW_COLLECTION}")


Đã tạo BoldSearcher_v3
